In [ ]:
!pip install transformers datasets torch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from PIL import Image
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    CLIPModel,
    CLIPProcessor
)
from datasets import load_dataset


# Load dataset and model
dataset = load_dataset("imdb")  # Example dataset
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Speed optimization 1: Freeze some layers to reduce training parameters
for param in model.distilbert.embeddings.parameters():
    param.requires_grad = False

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)  # Speed optimization 2: Limit max length

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(500))  # Speed optimization 3: Even smaller dataset
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(50))  # Reduced evaluation set

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Configuration for optimized training
training_args = TrainingArguments(
    output_dir="./results",  # Directory where model checkpoints and logs will be saved

    # Training strategy settings
    eval_strategy="no",  # Disables evaluation during training to save time (options: "no", "steps", "epoch")
    num_train_epochs=1,  # Number of training epochs - reduced to speed up training while still showing learning

    # Batch size configuration
    per_device_train_batch_size=16,  # Number of samples processed on each device during training - larger values use more memory but train faster
    per_device_eval_batch_size=16,  # Number of samples processed on each device during evaluation
    gradient_accumulation_steps=2,  # Accumulates gradients over multiple batches before updating weights - simulates larger batch sizes with limited memory

    # Optimization settings
    fp16=True,  # Uses 16-bit floating point precision instead of 32-bit - significantly speeds up training on compatible GPUs with minimal accuracy loss

    # Logging and reporting
    report_to="none",  # Disables external logging services (options: "all", "tensorboard", "wandb", "none")
    run_name="distilbert_imdb",  # Name of the training run for organization purposes
    logging_steps=10,  # Controls frequency of log updates - higher values reduce overhead of logging

    # Additional optimizations that could be added:
    # learning_rate=5e-5,  # Controls step size in weight updates - usually 1e-4 to 5e-5 for fine-tuning
    # weight_decay=0.01,  # Regularization to prevent overfitting
    # warmup_steps=100,  # Gradually increases learning rate at start of training to improve stability
)

In [ ]:
# Create trainer with train_dataset:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
    # eval_dataset removed since eval_strategy is "no"
)
trainer.train()

Step,Training Loss
10,0.689700


TrainOutput(global_step=16, training_loss=0.6800574213266373, metrics={'train_runtime': 333.9663, 'train_samples_per_second': 1.497, 'train_steps_per_second': 0.048, 'total_flos': 16558424832000.0, 'train_loss': 0.6800574213266373, 'epoch': 1.0})

In [ ]:
inputs = tokenizer("This movie was absolutely fantastic!", return_tensors="pt")
outputs = model(**inputs)
print(outputs.logits)

tensor([[ 0.1084, -0.0038]], grad_fn=<AddmmBackward0>)


In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
# INEFFICIENT: This approach has several problems:
# 1. Tokenizes each example individually inside the loop (expensive repeated computation)
# 2. Processes single examples instead of batches (very slow)
# 3. Doesn't handle device placement (GPU/CPU)
# 4. Doesn't use a proper DataLoader
# 5. Optimizer created outside the loop but not used consistently


In [ ]:
for epoch in range(5):  # INEFFICIENT: Too many epochs for small datasets
    for batch in train_dataset:
        # INEFFICIENT: Repeated tokenization inside loop for each example
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True)
        inputs['labels'] = torch.tensor([batch["label"]])

        # INEFFICIENT: No device management (CPU/GPU)
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

In [ ]:
# Fresh optimizer for this approach
optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
# EFFICIENT: Improvements in this approach:
# 1. Uses DataLoader for proper batching
# 2. Handles device placement (GPU/CPU)
# 3. Uses fewer epochs
# 4. Properly handles the optimizer

# Create DataLoader for efficient batching
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Fresh optimizer for this approach
optimizer = AdamW(model.parameters(), lr=5e-5)

# Determine device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Move model to appropriate device

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
for epoch in range(2):  # EFFICIENT: Reduced epochs
    for batch in train_dataloader:  # EFFICIENT: Proper batching
        # Process batch efficiently
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True)
        inputs['labels'] = batch["label"]

        # EFFICIENT: Move everything to the same device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Forward pass and loss calculation
        outputs = model(**inputs)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

In [ ]:
# Example of using CLIP for image-text similarity
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
import requests
from io import BytesIO

In [ ]:
# Loading image from URL instead of local file
# Insert a URL below
image_url = "https://dog.com/image.jpg"
try:
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content))
    inputs = processor(text=["a cat", "a dog"], images=image, return_tensors="pt", padding=True)
    outputs = model(**inputs)
    print(outputs.logits_per_image)  # Image-text similarity scores
    # [0.1231, 0.65]
except Exception as e:
    print(f"Error loading image from URL: {e}")

Error loading image from URL: Invalid URL '': No scheme supplied. Perhaps you meant https://?
